# 🤝 Colaboración entre 2 agentes IQL (DQN independientes) en un gridworld 3D

Dos agentes (cubos **amarillo** y **rosa**) viven en un tablero de **5×5** con **2 cubos verdes** y **2 pozos**.
Deben **empujar los cubos hasta los pozos**. Cada agente tiene **su propia red DQN**, su propio replay buffer y su propia
red objetivo (*Independent Q-Learning*): no comparten parámetros ni experiencia; el compañero es, para cada uno,
una parte más del entorno.

**Reglas**
- 4 acciones: `avanzar`, `retroceder`, `izquierda`, `derecha`.
- Caminar hacia un cubo lo empuja una celda (si la celda de destino está libre). Si cae en un pozo, lo rellena.
- Si un agente cae a un pozo o sale por el borde, recibe una recompensa negativa **solo él** y el otro puede seguir.
- Si se meten **todos** los cubos en los pozos, los agentes que siguen vivos reciben una recompensa **grande**.

**Recompensas (configurables en la celda 2)**

| Evento | Quién la recibe | Valor |
|---|---|---|
| Cada paso | cada agente vivo | -0.02 |
| Un cubo cae a un pozo | ambos (vivos) | +1 |
| **Todos** los cubos en pozos | ambos (vivos) | **+10** |
| Acción que no mueve (choca) | cada agente | -0.05 |
| Caer a un pozo / por el borde | solo el que cae | -5 |

**Observación** (cada agente ve todo el tablero, 4 planos de 5×5): su posición, la del compañero, los cubos y los pozos abiertos.

Cada `VIDEO_EVERY` episodios se muestra un video 3D de la política *greedy* actual, así se ve cómo van aprendiendo.
Corré las celdas en orden (Runtime → Run all). Con CPU alcanza.

In [ ]:
!pip -q install imageio imageio-ffmpeg

## 1. Configuración
Acá se cambia todo: tamaño de pasos, recompensas, hiperparámetros, frecuencia de los videos, velocidad (`FPS`, `SUBFRAMES`).

`LAYOUT_MODE`: `"fijo"` (siempre igual), `"agentes_aleatorios"` (cubos y pozos fijos, los agentes arrancan en lugares al azar; **recomendado**) o `"aleatorio"` (todo al azar, más difícil y más lento de aprender).

In [ ]:
from types import SimpleNamespace

CFG = SimpleNamespace(
    # ---------------- mundo ----------------
    GRID=5,
    MAX_STEPS=40,                 # pasos máximos por episodio
    LAYOUT_MODE="agentes_aleatorios",   # "fijo" | "agentes_aleatorios" | "aleatorio"
    # Layout fijo (x = columna, y = fila). Los videos siempre usan este layout.
    FIXED_LAYOUT=dict(agents=[(0, 0), (4, 4)], boxes=[(1, 2), (3, 2)], holes=[(3, 0), (1, 4)]),

    # ---------------- recompensas ----------------
    STEP_PENALTY=-0.02,           # costo por paso (individual)
    BOX_IN_HOLE_REWARD=1.0,       # por cada cubo que cae a un pozo (para los dos, si siguen vivos)
    SUCCESS_REWARD=10.0,          # recompensa grande al meter TODOS los cubos (para los dos)
    BLOCKED_PENALTY=-0.05,        # extra si la acción no mueve al agente (choca) -> evita quedarse quieto
    FALL_PENALTY=-5.0,            # caer a un pozo o por el borde (solo el que cae)

    # ---------------- DQN ----------------
    GAMMA=0.97,
    LR=1e-3,
    BATCH=64,
    BUFFER=50_000,
    HIDDEN=128,
    TARGET_SYNC=300,              # cada cuántos pasos de aprendizaje se copia la red objetivo
    LEARN_EVERY=1,
    WARMUP=500,
    EPS_START=1.0,
    EPS_END=0.05,
    EPS_DECAY_EPISODES=2000,

    # ---------------- entrenamiento / videos ----------------
    MAX_EPISODES=6000,
    MIN_EPISODES=2500,            # no se corta antes de este episodio (la curva se estabiliza)
    EVAL_EPS=0.05,                # ε en evaluaciones y videos (rompe bucles de política determinista)
    VIDEO_EVERY=250,              # cada cuántos episodios se muestra un video
    EVAL_EPISODES=30,
    STOP_SUCCESS=0.9,             # se corta cuando el éxito (greedy) supera este valor 2 evaluaciones seguidas
    FPS=14,
    SUBFRAMES=4,                  # cuadros por movimiento (más = más suave/lento)
    SEED=0,
)


## 2. Entorno (gridworld con empujes)

In [ ]:
import random
import numpy as np

# ---------------------------------------------------------------------
# Acciones: 0 = avanzar, 1 = retroceder, 2 = izquierda, 3 = derecha
# (en la vista isométrica: arriba-derecha, abajo-izquierda, arriba-izquierda, abajo-derecha)
# ---------------------------------------------------------------------
ACTION_NAMES = ["avanzar", "retroceder", "izquierda", "derecha"]
DELTAS = [(0, -1), (0, 1), (-1, 0), (1, 0)]  # (dx, dy)


class PushWorld:
    """Gridworld 5x5 con 2 agentes, 2 cubos y 2 pozos.

    - Un agente que camina hacia un cubo lo empuja una celda (si la celda de destino está libre).
    - Un cubo empujado a un pozo lo rellena y desaparece.
    - Un agente que entra a un pozo o sale del tablero cae: recompensa negativa SOLO para él,
      y el otro agente puede seguir jugando.
    - Cuando todos los cubos están en pozos, los agentes que siguen vivos reciben una recompensa grande.
    """

    def __init__(self, cfg):
        self.cfg = cfg
        self.N = cfg.GRID
        self.n_agents = 2
        self.n_actions = 4
        self.obs_dim = 4 * self.N * self.N
        self.rng = random.Random(cfg.SEED)

    # ------------------------------------------------------------------ layouts
    def _random_layout(self, randomize_all):
        N = self.N
        fixed = self.cfg.FIXED_LAYOUT
        if not randomize_all:
            return {k: list(v) for k, v in fixed.items()}
        cells = [(x, y) for x in range(N) for y in range(N)]
        inner = [(x, y) for x in range(1, N - 1) for y in range(1, N - 1)]
        boxes = self.rng.sample(inner, 2)
        rest = [c for c in cells if c not in boxes]
        holes = self.rng.sample(rest, 2)
        rest = [c for c in rest if c not in holes]
        agents = self.rng.sample(rest, 2)
        return {"agents": agents, "boxes": boxes, "holes": holes}

    def reset(self, layout=None):
        """layout=None -> usa cfg.LAYOUT_MODE. layout='fixed' fuerza el layout fijo (para los videos)."""
        mode = self.cfg.LAYOUT_MODE
        if layout == "fixed":
            lay = self._random_layout(False)
        elif layout is not None:
            lay = layout
        elif mode == "fijo":
            lay = self._random_layout(False)
        elif mode == "agentes_aleatorios":
            lay = self._random_layout(False)
            N = self.N
            taken = set(map(tuple, lay["boxes"])) | set(map(tuple, lay["holes"]))
            free = [(x, y) for x in range(N) for y in range(N) if (x, y) not in taken]
            lay["agents"] = self.rng.sample(free, 2)
        else:
            lay = self._random_layout(True)
        self.agents = [tuple(p) for p in lay["agents"]]
        self.boxes = [tuple(p) for p in lay["boxes"]]
        self.holes = [tuple(p) for p in lay["holes"]]
        self.filled = [False, False]
        self.alive = [True, True]
        self.status = ["ok", "ok"]
        self.t = 0
        self.total_reward = [0.0, 0.0]
        self.success = False
        self.anim = None
        return self.observe()

    # ------------------------------------------------------------------ observaciones
    def observe(self):
        """Cada agente ve el mundo completo (4 planos NxN): yo, el otro, cubos, pozos abiertos."""
        N = self.N
        boxes = np.zeros((N, N), np.float32)
        holes = np.zeros((N, N), np.float32)
        for b in self.boxes:
            if b is not None:
                boxes[b[1], b[0]] = 1
        for h, f in zip(self.holes, self.filled):
            if not f:
                holes[h[1], h[0]] = 1
        obs = []
        for i in range(2):
            me = np.zeros((N, N), np.float32)
            other = np.zeros((N, N), np.float32)
            if self.alive[i]:
                me[self.agents[i][1], self.agents[i][0]] = 1
            j = 1 - i
            if self.alive[j]:
                other[self.agents[j][1], self.agents[j][0]] = 1
            obs.append(np.concatenate([me.ravel(), other.ravel(), boxes.ravel(), holes.ravel()]))
        return obs

    # ------------------------------------------------------------------ helpers
    def _inside(self, x, y):
        return 0 <= x < self.N and 0 <= y < self.N

    def _box_at(self, x, y):
        for k, b in enumerate(self.boxes):
            if b == (x, y):
                return k
        return None

    def _open_hole_at(self, x, y):
        for k, h in enumerate(self.holes):
            if h == (x, y) and not self.filled[k]:
                return k
        return None

    # ------------------------------------------------------------------ paso
    def step(self, actions):
        """actions: lista de 2 enteros (se ignora la de un agente muerto).
        Devuelve obs, rewards, dones(por agente), episode_over, info."""
        c = self.cfg
        self.t += 1
        was_alive = list(self.alive)
        rewards = [0.0, 0.0]
        filled_before = list(self.filled)

        # info para animar: posición inicial/final de cada entidad en este paso
        A = [dict(start=self.agents[i], end=self.agents[i], fall=None, cell=None, gone=not self.alive[i])
             for i in range(2)]
        B = [dict(start=b, end=b, fall=None, cell=None, gone=(b is None)) for b in self.boxes]

        start_pos = list(self.agents)
        order = [0, 1]
        self.rng.shuffle(order)
        boxes_dropped = 0
        for i in order:
            if not self.alive[i]:
                continue
            rewards[i] += c.STEP_PENALTY
            dx, dy = DELTAS[actions[i]]
            x, y = self.agents[i]
            nx, ny = x + dx, y + dy

            if not self._inside(nx, ny):                    # cae por el borde
                self.alive[i] = False
                self.status[i] = "borde"
                rewards[i] += c.FALL_PENALTY
                A[i].update(end=(nx, ny), fall="edge")
                continue
            j = 1 - i
            if self.alive[j] and self.agents[j] == (nx, ny):  # choca con el otro agente
                continue
            k = self._box_at(nx, ny)
            if k is not None:                                # empuja un cubo
                bx, by = nx + dx, ny + dy
                if not self._inside(bx, by):
                    continue
                if self._box_at(bx, by) is not None:
                    continue
                if self.alive[j] and self.agents[j] == (bx, by):
                    continue
                h = self._open_hole_at(bx, by)
                if h is not None:                            # el cubo cae al pozo
                    self.filled[h] = True
                    self.boxes[k] = None
                    boxes_dropped += 1
                    B[k].update(end=(bx, by), fall="hole", cell=(bx, by))
                else:
                    self.boxes[k] = (bx, by)
                    B[k].update(end=(bx, by))
                self.agents[i] = (nx, ny)
                A[i].update(end=(nx, ny))
                continue
            # celda libre (o pozo abierto)
            self.agents[i] = (nx, ny)
            A[i].update(end=(nx, ny))
            if self._open_hole_at(nx, ny) is not None:       # cae al pozo
                self.alive[i] = False
                self.status[i] = "pozo"
                rewards[i] += c.FALL_PENALTY
                A[i].update(fall="hole", cell=(nx, ny))

        for i in range(2):                                   # chocar contra pared/cubo/compañero
            if was_alive[i] and self.alive[i] and self.agents[i] == start_pos[i]:
                rewards[i] += c.BLOCKED_PENALTY

        # recompensas de equipo (solo para quien sigue vivo)
        all_done = all(b is None for b in self.boxes)
        for i in range(2):
            if self.alive[i]:
                rewards[i] += c.BOX_IN_HOLE_REWARD * boxes_dropped
                if all_done:
                    rewards[i] += c.SUCCESS_REWARD
        self.success = all_done
        for i in range(2):
            if was_alive[i]:
                self.total_reward[i] += rewards[i]

        terminated = all_done or not any(self.alive)
        truncated = self.t >= c.MAX_STEPS
        # "done" por agente para el target de DQN: cae o termina el episodio
        dones = [(was_alive[i] and (not self.alive[i] or terminated)) for i in range(2)]
        self.anim = dict(agents=A, boxes=B, filled_before=filled_before, filled_after=list(self.filled))
        info = dict(success=self.success, truncated=truncated and not terminated, was_alive=was_alive)
        return self.observe(), rewards, dones, (terminated or truncated), info


## 3. Render 3D isométrico
Renderizado propio con PIL (sin dependencias raras): losas azules, pozos oscuros, cubos verdes y agentes con orugas. Las caídas y los empujes se animan.

In [ ]:
import os, math
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageChops
import matplotlib

# ---------------- paleta (inspirada en la imagen de referencia) ----------------
BG_TOP, BG_BOT = (22, 52, 84), (14, 34, 58)
TILE_TOP, TILE_L, TILE_R = (74, 144, 192), (33, 75, 105), (55, 112, 150)
FILLED_TOP = (96, 178, 170)
VOID = (10, 26, 44)
PIT_L, PIT_R = (40, 92, 128), (28, 66, 94)
BOX_C = dict(top=(93, 196, 170), left=(42, 96, 80), right=(66, 142, 120))
AG_C = [dict(top=(255, 255, 20), left=(190, 160, 10), right=(225, 205, 10), panel=(255, 235, 90), name="Amarillo"),
        dict(top=(255, 90, 215), left=(170, 55, 145), right=(215, 70, 185), panel=(244, 75, 200), name="Rosa")]
TRACK = dict(top=(95, 95, 95), left=(55, 55, 55), right=(75, 75, 75))

W, H = 720, 480      # tamaño final del cuadro
SS = 2               # supersampling (anti-aliasing)


def _font(size):
    try:
        p = os.path.join(matplotlib.get_data_path(), "fonts/ttf/DejaVuSans-Bold.ttf")
        return ImageFont.truetype(p, size)
    except Exception:
        return ImageFont.load_default()


class IsoRenderer:
    def __init__(self, grid=5):
        self.N = grid
        self.a = 60 * SS          # semi-ancho de una celda en pantalla
        self.b = 34 * SS          # semi-alto
        self.zu = 56 * SS         # px por "unidad" de altura (1 celda de alto)
        self.T = 34 * SS          # espesor de la losa
        self.ox = W * SS / 2
        self.oy = 92 * SS
        self.f_big, self.f_small = _font(22), _font(15)
        self.bg = self._make_bg()

    def _make_bg(self):
        h = np.linspace(0, 1, H)[:, None, None]
        top, bot = np.array(BG_TOP)[None, None, :], np.array(BG_BOT)[None, None, :]
        arr = (top * (1 - h) + bot * h) * np.ones((H, W, 3))
        return Image.fromarray(arr.astype(np.uint8)).resize((W * SS, H * SS))

    # proyección: coordenadas de grilla (gx, gy) + altura z (px) -> pantalla
    def P(self, gx, gy, z=0.0):
        return (self.ox + (gx - gy) * self.a, self.oy + (gx + gy) * self.b - z)

    def _poly(self, d, pts, fill):
        d.polygon(pts, fill=fill)

    # --------------------------------------------------------------- piso
    def _draw_floor(self, d, img, holes, filled):
        N, T = self.N, self.T
        hole_at = {h: k for k, h in enumerate(holes)}
        for s in range(2 * N - 1):
            for x in range(N):
                y = s - x
                if not (0 <= y < N):
                    continue
                if (x, y) in hole_at and not filled[hole_at[(x, y)]]:
                    self._draw_pit(img, x, y, holes, filled)
                    continue
                top = FILLED_TOP if (x, y) in hole_at else TILE_TOP
                p0, p1, p2, p3 = self.P(x, y), self.P(x + 1, y), self.P(x + 1, y + 1), self.P(x, y + 1)
                # caras laterales: solo si el vecino (frontal) no existe como losa
                def is_tile(cx, cy):
                    if not (0 <= cx < N and 0 <= cy < N):
                        return False
                    return not ((cx, cy) in hole_at and not filled[hole_at[(cx, cy)]])
                if not is_tile(x, y + 1):
                    self._poly(d, [p3, p2, (p2[0], p2[1] + T), (p3[0], p3[1] + T)], TILE_L)
                if not is_tile(x + 1, y):
                    self._poly(d, [p2, p1, (p1[0], p1[1] + T), (p2[0], p2[1] + T)], TILE_R)
                self._poly(d, [p0, p1, p2, p3], top)

    def _draw_pit(self, img, x, y, holes, filled):
        """Pozo: rombo oscuro con las paredes internas visibles, recortado al rombo."""
        N, T = self.N, self.T
        layer = Image.new("RGBA", img.size, (0, 0, 0, 0))
        ld = ImageDraw.Draw(layer)
        t, r, bm, l = self.P(x, y), self.P(x + 1, y), self.P(x + 1, y + 1), self.P(x, y + 1)
        dd = T * 1.0
        ld.polygon([t, r, bm, l], fill=VOID + (255,))

        def tile_exists(cx, cy):
            if not (0 <= cx < N and 0 <= cy < N):
                return False
            for k, h in enumerate(holes):
                if h == (cx, cy):
                    return filled[k]
            return True
        if tile_exists(x, y - 1):   # pared trasera derecha (cara +y del vecino de atrás)
            ld.polygon([t, r, (r[0], r[1] + dd), (t[0], t[1] + dd)], fill=PIT_L + (255,))
        if tile_exists(x - 1, y):   # pared trasera izquierda (cara +x del vecino)
            ld.polygon([t, l, (l[0], l[1] + dd), (t[0], t[1] + dd)], fill=PIT_R + (255,))
        mask = Image.new("L", img.size, 0)
        ImageDraw.Draw(mask).polygon([t, r, bm, l], fill=255)
        img.paste(layer, (0, 0), ImageChops.multiply(layer.split()[3], mask))

    # --------------------------------------------------------------- cubos
    def _cube(self, d, cx, cy, zb, size, height, col):
        """cubo centrado en (cx, cy) coordenadas de grilla (continuas), base a altura zb px."""
        s = size / 2
        zt = zb + height
        T_, R_, B_, L_ = (cx - s, cy - s), (cx + s, cy - s), (cx + s, cy + s), (cx - s, cy + s)
        pl_b, pb_b, pr_b = self.P(*L_, zb), self.P(*B_, zb), self.P(*R_, zb)
        pl_t, pb_t, pr_t, pt_t = self.P(*L_, zt), self.P(*B_, zt), self.P(*R_, zt), self.P(*T_, zt)
        self._poly(d, [pl_t, pb_t, pb_b, pl_b], col["left"])
        self._poly(d, [pb_t, pr_t, pr_b, pb_b], col["right"])
        self._poly(d, [pt_t, pr_t, pb_t, pl_t], col["top"])
        return pl_t, pb_t, pr_t, pt_t, pl_b, pb_b

    def _draw_agent(self, d, cx, cy, z, colr, facing):
        sz = 0.62
        self._cube(d, cx, cy, z, sz, 0.14 * self.zu, TRACK)                      # orugas
        pl_t, pb_t, pr_t, pt_t, pl_b, pb_b = self._cube(d, cx, cy, z + 0.14 * self.zu, 0.54, 0.52 * self.zu, colr)
        # ojitos en la cara izquierda y derecha
        def eye(p_from, p_to, u, v):
            ex = p_from[0] + (p_to[0] - p_from[0]) * u
            ey = p_from[1] + (p_to[1] - p_from[1]) * u + v
            r = 0.035 * self.zu
            d.rectangle([ex - r, ey - r, ex + r, ey + r], fill=(40, 40, 40))
        for u in (0.25, 0.65):
            eye(pl_t, pb_t, u, 0.16 * self.zu)

    def _draw_box(self, d, cx, cy, z):
        self._cube(d, cx, cy, z, 0.72, 0.72 * self.zu, BOX_C)

    # --------------------------------------------------------------- cuadro
    def frame(self, env, tt, scores, status_text="", ep_text=""):
        """tt en [0,1]: avance de la animación del último paso (env.anim). tt=None -> estado actual."""
        N = self.N
        img = self.bg.copy()
        d = ImageDraw.Draw(img)
        anim = env.anim
        if tt is None and anim is not None:
            tt = 1.0                  # estado final del paso: lo que cayó no reaparece
        filled = env.filled if (anim is None or tt is None or tt >= 1) else anim["filled_before"]
        self._draw_floor(d, img, env.holes, filled)

        ents = []   # (orden, tipo, datos)
        move_end = 0.55

        def pos_of(e):
            if anim is None or tt is None:
                return e["end"][0] + 0.5, e["end"][1] + 0.5, 0.0, 1.0
            u = min(1.0, tt / move_end)
            u = u * u * (3 - 2 * u)
            x = e["start"][0] + (e["end"][0] - e["start"][0]) * u
            y = e["start"][1] + (e["end"][1] - e["start"][1]) * u
            z = 0.0
            if e["fall"] and tt > move_end:
                f = (tt - move_end) / (1 - move_end)
                z = -(f * f) * 2.2 * self.zu
            return x + 0.5, y + 0.5, z, 1.0

        if anim is None:
            for k, b in enumerate(env.boxes):
                if b is not None:
                    ents.append((b[0] + b[1] + 1, "box", dict(x=b[0] + .5, y=b[1] + .5, z=0, fall=None, cell=None)))
            for i in range(2):
                if env.alive[i]:
                    p = env.agents[i]
                    ents.append((p[0] + p[1] + 1, "ag", dict(i=i, x=p[0] + .5, y=p[1] + .5, z=0, fall=None, cell=None)))
        else:
            for k, e in enumerate(anim["boxes"]):
                if e["gone"]:
                    continue
                x, y, z, _ = pos_of(e)
                if e["fall"] and tt is not None and tt >= 1:
                    continue
                ents.append((x + y, "box", dict(x=x, y=y, z=z, fall=e["fall"], cell=e["cell"])))
            for i, e in enumerate(anim["agents"]):
                if e["gone"]:
                    continue
                x, y, z, _ = pos_of(e)
                if e["fall"] and tt is not None and tt >= 1:
                    continue
                ents.append((x + y + 0.01, "ag", dict(i=i, x=x, y=y, z=z, fall=e["fall"], cell=e["cell"])))
        ents.sort(key=lambda e: e[0])

        for _, kind, e in ents:
            if e["fall"] == "hole":
                # dibujar en capa aparte y recortar: lo que cae "desaparece" dentro del pozo
                layer = Image.new("RGBA", img.size, (0, 0, 0, 0))
                ld = ImageDraw.Draw(layer)
                (self._draw_box(ld, e["x"], e["y"], e["z"]) if kind == "box"
                 else self._draw_agent(ld, e["x"], e["y"], e["z"], AG_C[e["i"]], None))
                hx, hy = e["cell"]
                l, bm, r = self.P(hx, hy + 1), self.P(hx + 1, hy + 1), self.P(hx + 1, hy)
                mask = Image.new("L", img.size, 0)
                ImageDraw.Draw(mask).polygon([l, bm, r, (r[0], 0), (l[0], 0)], fill=255)
                a = layer.split()[3]
                img.paste(layer, (0, 0), ImageChops.multiply(a, mask))
                d = ImageDraw.Draw(img)
            else:
                (self._draw_box(d, e["x"], e["y"], e["z"]) if kind == "box"
                 else self._draw_agent(d, e["x"], e["y"], e["z"], AG_C[e["i"]], None))

        out = img.resize((W, H), Image.LANCZOS)
        self._hud(out, scores, status_text, ep_text, env)
        return np.asarray(out)

    def _hud(self, out, scores, status_text, ep_text, env):
        d = ImageDraw.Draw(out)
        d.text((16, 12), ep_text, font=self.f_big, fill=(235, 242, 250))
        d.text((16, 42), status_text, font=self.f_small, fill=(150, 190, 225))
        for i, (x0, anchor) in enumerate([(14, "l"), (W - 14 - 196, "r")]):
            col = AG_C[i]["panel"]
            d.rectangle([x0, H - 88, x0 + 196, H - 12], fill=col)
            st = {"ok": "", "pozo": "  (cayó al pozo)", "borde": "  (cayó al borde)"}[env.status[i]]
            d.text((x0 + 10, H - 84), AG_C[i]["name"], font=self.f_small, fill=(40, 30, 50))
            d.text((x0 + 10, H - 62), "R = %+.2f" % scores[i], font=self.f_big, fill=(30, 20, 40))
            if st:
                d.text((x0 + 10, H - 32), st.strip(), font=self.f_small, fill=(90, 20, 30))


In [ ]:
# Vista previa del mundo
from IPython.display import display
from PIL import Image
_e = PushWorld(CFG); _e.reset("fixed")
display(Image.fromarray(IsoRenderer(CFG.GRID).frame(_e, None, [0, 0], "vista previa", "Mundo 5x5")))

## 4. Agentes DQN independientes (IQL)

In [ ]:
import copy, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cpu")   # la red es chica: CPU alcanza y sobra
torch.set_num_threads(2)


class QNet(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


class DQNAgent:
    """Un agente DQN completamente independiente (IQL): su red, su target, su replay buffer.
    No comparte parámetros ni experiencia con el otro agente; para él, el compañero es parte del entorno."""

    def __init__(self, obs_dim, n_actions, cfg, seed):
        self.cfg, self.n_actions = cfg, n_actions
        torch.manual_seed(seed)
        self.q = QNet(obs_dim, n_actions, cfg.HIDDEN).to(DEVICE)
        self.target = copy.deepcopy(self.q)
        self.opt = torch.optim.Adam(self.q.parameters(), lr=cfg.LR)
        n = cfg.BUFFER
        self.S = np.zeros((n, obs_dim), np.float32)
        self.S2 = np.zeros((n, obs_dim), np.float32)
        self.A = np.zeros(n, np.int64)
        self.R = np.zeros(n, np.float32)
        self.D = np.zeros(n, np.float32)
        self.size = self.ptr = self.learn_steps = 0
        self.rng = np.random.default_rng(seed)
        self.last_loss = float("nan")

    @torch.no_grad()
    def act(self, obs, eps):
        if self.rng.random() < eps:
            return int(self.rng.integers(self.n_actions))
        x = torch.from_numpy(obs).unsqueeze(0).to(DEVICE)
        return int(self.q(x).argmax(dim=1).item())

    def remember(self, s, a, r, s2, done):
        i = self.ptr
        self.S[i], self.A[i], self.R[i], self.S2[i], self.D[i] = s, a, r, s2, float(done)
        self.ptr = (i + 1) % self.cfg.BUFFER
        self.size = min(self.size + 1, self.cfg.BUFFER)

    def learn(self):
        c = self.cfg
        if self.size < max(c.WARMUP, c.BATCH):
            return
        idx = self.rng.integers(0, self.size, c.BATCH)
        s = torch.from_numpy(self.S[idx]).to(DEVICE)
        s2 = torch.from_numpy(self.S2[idx]).to(DEVICE)
        a = torch.from_numpy(self.A[idx]).to(DEVICE)
        r = torch.from_numpy(self.R[idx]).to(DEVICE)
        d = torch.from_numpy(self.D[idx]).to(DEVICE)
        q = self.q(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            y = r + c.GAMMA * (1.0 - d) * self.target(s2).max(dim=1).values
        loss = F.smooth_l1_loss(q, y)
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.opt.step()
        self.last_loss = float(loss.item())
        self.learn_steps += 1
        if self.learn_steps % c.TARGET_SYNC == 0:
            self.target.load_state_dict(self.q.state_dict())


## 5. Entrenamiento con videos periódicos
Se muestra un video del episodio 0 (agentes sin entrenar) y luego uno cada `VIDEO_EVERY` episodios.
Los videos usan siempre el mismo layout fijo y la política *greedy*, para poder comparar. El entrenamiento corta solo cuando
el éxito *greedy* supera `STOP_SUCCESS` en dos evaluaciones seguidas.

In [ ]:
import base64, io, time
import numpy as np
import imageio
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#2b2b2b', 'axes.facecolor': '#2b2b2b', 'axes.edgecolor': '#cccccc',
    'axes.labelcolor': '#cccccc', 'xtick.color': '#cccccc', 'ytick.color': '#cccccc',
    'text.color': '#cccccc', 'grid.color': '#444444',
})
from IPython.display import HTML, display, clear_output

random.seed(CFG.SEED); np.random.seed(CFG.SEED)
env = PushWorld(CFG)
renderer = IsoRenderer(CFG.GRID)
agents = [DQNAgent(env.obs_dim, env.n_actions, CFG, CFG.SEED + 11 * i) for i in range(2)]


def greedy_actions(obs, eps=0.0, memo=None):
    """Acciones greedy (con ε chico). Si el estado se repite (agentes quietos u oscilando),
    el agente toma una acción al azar para salir del bucle."""
    acts = []
    for i in range(2):
        if not env.alive[i]:
            acts.append(0); continue
        a = agents[i].act(obs[i], eps)
        if memo is not None:
            key = obs[i].tobytes()
            if key in memo[i][-4:]:
                a = int(np.random.randint(env.n_actions))
            memo[i].append(key)
        acts.append(a)
    return acts


def evaluate(n_episodes):
    """Éxito y recompensa media con política greedy (sin exploración)."""
    wins, rets = 0, []
    for _ in range(n_episodes):
        obs = env.reset()
        memo = [[], []]
        for _ in range(CFG.MAX_STEPS):
            obs, r, d, over, info = env.step(greedy_actions(obs, CFG.EVAL_EPS, memo))
            if over:
                break
        wins += int(env.success)
        rets.append(sum(env.total_reward))
    return wins / n_episodes, float(np.mean(rets))


def record_episode(title, eps=CFG.EVAL_EPS, max_steps=None, layout="fixed"):
    """Juega UN episodio (por defecto con el layout fijo) y devuelve los cuadros del video."""
    obs = env.reset(layout)
    frames = []
    memo = [[], []]
    sc = env.total_reward
    for _ in range(6):                       # un instante para ver el estado inicial
        frames.append(renderer.frame(env, None, sc, "estado inicial", title))
    for step in range(max_steps or CFG.MAX_STEPS):
        obs, r, d, over, info = env.step(greedy_actions(obs, eps, memo))
        for k in range(1, CFG.SUBFRAMES + 1):
            frames.append(renderer.frame(env, k / CFG.SUBFRAMES, sc, f"paso {env.t}", title))
        if over:
            break
    msg = "¡Los dos cubos están en los pozos!" if env.success else (
        "Ambos agentes cayeron" if not any(env.alive) else "Se acabó el tiempo")
    for _ in range(int(CFG.FPS * 1.5)):
        frames.append(renderer.frame(env, None, sc, msg, title))
    return frames, env.success


def show_video(frames, width=520):
    try:
        path = "/tmp/_ep.mp4"
        imageio.mimwrite(path, frames, fps=CFG.FPS, codec="libx264", quality=7, macro_block_size=16)
        b64 = base64.b64encode(open(path, "rb").read()).decode()
        display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                     f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
    except Exception as e:                    # plan B: GIF
        from PIL import Image
        path = "/tmp/_ep.gif"
        ims = [Image.fromarray(f) for f in frames[::2]]
        ims[0].save(path, save_all=True, append_images=ims[1:], duration=int(2000 / CFG.FPS), loop=0)
        b64 = base64.b64encode(open(path, "rb").read()).decode()
        display(HTML(f'<img width="{width}" src="data:image/gif;base64,{b64}">'))


def epsilon(ep):
    f = min(1.0, ep / CFG.EPS_DECAY_EPISODES)
    return CFG.EPS_START + (CFG.EPS_END - CFG.EPS_START) * f


hist = dict(ep=[], train_ret=[], eval_ep=[], eval_succ=[], eval_ret=[])
ret_window = []
good_evals = 0
t0 = time.time()

# --- video del episodio 0: agentes sin entrenar (política casi aleatoria) ---
print("▶ Episodio 0 — agentes sin entrenar")
fr, ok = record_episode("Episodio 0 (sin entrenar)")
show_video(fr)

for ep in range(1, CFG.MAX_EPISODES + 1):
    eps = epsilon(ep)
    obs = env.reset()
    for _ in range(CFG.MAX_STEPS):
        alive = list(env.alive)
        acts = [agents[i].act(obs[i], eps) if alive[i] else 0 for i in range(2)]
        obs2, rew, dones, over, info = env.step(acts)
        for i in range(2):
            if alive[i]:
                agents[i].remember(obs[i], acts[i], rew[i], obs2[i], dones[i])   # solo su propia experiencia
            agents[i].learn()                                                   # cada uno aprende por separado
        obs = obs2
        if over:
            break
    ret_window.append(sum(env.total_reward))
    hist["ep"].append(ep); hist["train_ret"].append(sum(env.total_reward))

    if ep % CFG.VIDEO_EVERY == 0:
        succ, ret = evaluate(CFG.EVAL_EPISODES)
        hist["eval_ep"].append(ep); hist["eval_succ"].append(succ); hist["eval_ret"].append(ret)
        print(f"▶ Episodio {ep} | ε={eps:.2f} | éxito greedy={succ:.0%} | retorno medio={ret:+.2f} "
              f"| retorno entrenamiento (últ. {CFG.VIDEO_EVERY})={np.mean(ret_window[-CFG.VIDEO_EVERY:]):+.2f} "
              f"| {time.time()-t0:.0f}s")
        fr, ok = record_episode(f"Episodio {ep} (éxito {succ:.0%})")
        show_video(fr)
        good_evals = good_evals + 1 if succ >= CFG.STOP_SUCCESS else 0
        if good_evals >= 2 and ep >= CFG.MIN_EPISODES:
            print(f"✅ Los dos agentes aprendieron a colaborar (éxito ≥ {CFG.STOP_SUCCESS:.0%} en 2 evaluaciones seguidas).")
            break

# ---- curvas de aprendizaje ----
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
w = 100
tr = np.array(hist["train_ret"])
if len(tr) >= w:
    ax[0].plot(np.convolve(tr, np.ones(w) / w, mode="valid"))
ax[0].set_title("Retorno de equipo por episodio (media móvil 100)"); ax[0].set_xlabel("episodio"); ax[0].grid(alpha=.3)
ax[1].plot(hist["eval_ep"], hist["eval_succ"], "o-")
ax[1].set_ylim(-0.02, 1.02); ax[1].set_title("Tasa de éxito (política greedy)"); ax[1].set_xlabel("episodio"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. Probar lo aprendido con posiciones iniciales al azar
Varios episodios *greedy* con los agentes arrancando en lugares distintos.

In [ ]:
for k in range(3):
    fr, ok = record_episode(f"Prueba {k+1} (posiciones al azar)", eps=0.0, layout=None)
    show_video(fr)

## Ideas para experimentar
- `BOX_IN_HOLE_REWARD = 0` deja solo la recompensa grande final: aprenden igual pero mucho más lento.
- `LAYOUT_MODE = "aleatorio"` obliga a generalizar (hacen falta más episodios).
- Subir `FALL_PENALTY` o `STEP_PENALTY` cambia cuánto riesgo toman.
- `SUBFRAMES` y `FPS` controlan la velocidad del video.